# Fetch Macroeconomic and Yield Curve Data

This notebook handles fetching the necessary FRED data and saving it to the `data/` folder as CSV files to avoid pulling from the internet during every model execution.

In [1]:
import pandas as pd
import pandas_datareader.data as web
import os

# Ensure data directory exists
os.makedirs('../data', exist_ok=True)

# Define data parameters
START_DATE = '1980-01-01'
END_DATE = '2025-12-31'

YIELD_TICKERS = ['DGS1MO', 'DGS3MO', 'DGS6MO', 'DGS1', 'DGS2', 'DGS3', 'DGS5', 'DGS7', 'DGS10', 'DGS20', 'DGS30']

# Expanded macro panel (~40 series, aligned with ECB WP 544 data-rich approach)
MACRO_TICKERS = {
    # --- Prices / Inflation ---
    "inflation":            "CPIAUCSL",    # CPI headline
    "inflation_ex":         "CPILFESL",    # Core CPI
    "pce":                  "PCECTPI",     # PCE deflator
    "pce_core":             "PCEPILFE",    # Core PCE
    "ppi_all":              "PPIACO",      # PPI all commodities
    "ppi_finished":         "PPIFGS",      # PPI finished goods
    "oil_price":            "DCOILWTICO",  # WTI crude oil (monthly avg)
    "inf_exp_5y":           "T5YIE",       # 5Y breakeven inflation
    "inf_exp_consumer":     "MICH",        # U-Mich consumer inflation expectations
    # --- Labor Market ---
    "fedfunds":             "FEDFUNDS",    # Federal funds rate (target)
    "fedfunds_eff":         "DFF",         # Effective fed funds rate
    "unemployment":         "UNRATE",      # Unemployment rate
    "payroll":              "PAYEMS",      # Nonfarm payroll
    "unemp_duration":       "UEMPMEAN",    # Mean duration of unemployment
    "unemp_prime":          "LNS14000025", # Prime-age (25-54) unemployment
    "init_claims":          "ICSA",        # Initial jobless claims
    "avg_hours":            "AWHNONAG",    # Avg weekly hours, nonfarm
    # --- Output / Activity ---
    "output":               "INDPRO",      # Industrial production
    "capacity_util":        "CUMFNS",      # Capacity utilization, manufacturing
    "durable_orders":       "DGORDER",     # Durable goods orders
    "retail_sales":         "RSXFS",       # Retail sales ex food services
    "housing":              "HOUST",       # Housing starts
    "building_permits":     "PERMIT",      # Building permits
    # --- Surveys / Sentiment ---
    "consumer_sentiment":   "UMCSENT",     # U of Michigan consumer sentiment
    # --- Financial / Monetary ---
    "money_m1":             "M1SL",        # M1 money supply
    "money_m2":             "M2SL",        # M2 money supply
    "monetary_base":        "BOGMBASE",    # Monetary base
    "equity_market":        "SP500",       # S&P 500
    "volatility":           "VIXCLS",      # VIX
    "hy_spread":            "BAMLH0A0HYM2",# High-yield spread
    "baa_spread":           "BAA10YM",     # Baa-10Y Treasury spread
    "yield_slope_10_2":     "T10Y2Y",      # 10Y-2Y yield spread
    "yield_slope_10_3m":    "T10Y3M",      # 10Y-3M yield spread
    "mortgage_rate":        "MORTGAGE30US",# 30Y mortgage rate
    "usd_index":            "DTWEXBGS",    # Trade-weighted USD (broad)
}


In [2]:
def fetch_fred_data(tickers, start, end):
    if isinstance(tickers, dict):
        ticker_list = list(tickers.values())
    else:
        ticker_list = tickers
    try:
        data = web.DataReader(ticker_list, 'fred', start, end)
    except Exception as e:
        print(f"Warning: Error fetching grouped data: {e}")
        data_dict = {}
        for ticker in ticker_list:
            try:
                data_dict[ticker] = web.DataReader(ticker, 'fred', start, end)
            except:
                print(f"  Skipping {ticker}")
                continue
        if data_dict:
            data = pd.concat(data_dict, axis=1)
        else:
            raise e
    return data

print("Fetching extended yield curve data...")
yields_raw = fetch_fred_data(YIELD_TICKERS, START_DATE, END_DATE)

print("Fetching extended macroeconomic data...")
macro_raw = fetch_fred_data(MACRO_TICKERS, START_DATE, END_DATE)

Fetching extended yield curve data...
Fetching extended macroeconomic data...


In [3]:
# Save the data to the data folder
yields_file = '../data/yields_raw.csv'
macro_file = '../data/macro_raw.csv'

yields_raw.to_csv(yields_file)
macro_raw.to_csv(macro_file)

print("Data fetching and saving process completed successfully!")
print(f"Saved yields data to {yields_file}")
print(f"Saved macro data to {macro_file}")

Data fetching and saving process completed successfully!
Saved yields data to ../data/yields_raw.csv
Saved macro data to ../data/macro_raw.csv


---
## GSW Zero-Coupon Yield Curve Data (Federal Reserve Svensson Parameters)

Downloads the Gürkaynak-Sack-Wright (2006) nominal yield curve parameter file from the Federal Reserve website.  
The file contains daily **Svensson (Nelson-Siegel-Svensson)** parameters `BETA0–BETA3`, `TAU1`, `TAU2` and  
pre-computed zero-coupon spot yields `SVENY01–SVENY30`.

We apply the NSS formula to compute yields at our 11 target maturities (1M–30Y) and save:
- `data/yields_gsw_daily.csv` — daily, columns 1M–30Y  
- `data/yields_gsw_monthly.csv` — month-end mean, columns 1M–30Y (same convention as `yields_raw.csv`)

In [1]:
import os
import requests
import numpy as np
import pandas as pd

# ── Constants ─────────────────────────────────────────────────────────────────
GSW_URL         = "https://www.federalreserve.gov/data/yield-curve-tables/feds200628.csv"
GSW_LOCAL       = "../data/feds200628.csv"
GSW_DAILY_OUT   = "../data/yields_gsw_daily.csv"
GSW_MONTHLY_OUT = "../data/yields_gsw_monthly.csv"

GSW_MATURITIES  = [1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30]
GSW_LABELS      = ['1M', '3M', '6M', '1Y', '2Y', '3Y', '5Y', '7Y', '10Y', '20Y', '30Y']
GSW_START       = '1980-01-01'
GSW_END         = '2025-12-31'

# ── Download if not already cached ───────────────────────────────────────────
if not os.path.exists(GSW_LOCAL):
    print(f"Downloading GSW parameter file from Federal Reserve...")
    r = requests.get(GSW_URL, timeout=120)
    r.raise_for_status()
    with open(GSW_LOCAL, 'wb') as f:
        f.write(r.content)
    print(f"Saved to {GSW_LOCAL}")
else:
    print(f"Using cached GSW file: {GSW_LOCAL}")

# ── Parse: skip 9-row text preamble ──────────────────────────────────────────
# Row 0-8 are metadata text; row 9 is the column header.
# TAU2 = -999.99 is a sentinel meaning Nelson-Siegel (3-param, no BETA3 term).
gsw_raw = pd.read_csv(
    GSW_LOCAL,
    skiprows=9,
    parse_dates=['Date'],
    index_col='Date',
    na_values=['NA', ''],
)
gsw_raw['TAU2'] = gsw_raw['TAU2'].replace(-999.99, np.nan)

print(f"\nGSW raw shape : {gsw_raw.shape}")
print(f"Date range    : {gsw_raw.index.min().date()} – {gsw_raw.index.max().date()}")
print(f"Param columns : {[c for c in gsw_raw.columns if c.startswith('BETA') or c.startswith('TAU')]}")
print(f"SVENY columns : {[c for c in gsw_raw.columns if c.startswith('SVENY')][:5]} ...")

Using cached GSW file: ../data/feds200628.csv

GSW raw shape : (16923, 99)
Date range    : 1961-06-14 – 2026-04-24
Param columns : ['BETA0', 'BETA1', 'BETA2', 'BETA3', 'TAU1', 'TAU2']
SVENY columns : ['SVENY01', 'SVENY02', 'SVENY03', 'SVENY04', 'SVENY05'] ...


In [2]:
def svensson_yield(tau, b0, b1, b2, b3, t1, t2):
    """
    Nelson-Siegel-Svensson zero-coupon yield (continuously-compounded).

    Formula (per task spec):
        x1 = tau / TAU1,  x2 = tau / TAU2
        A  = (1 - exp(-x1)) / x1
        B  = A - exp(-x1)
        C  = (1 - exp(-x2)) / x2 - exp(-x2)
        y  = BETA0 + BETA1*A + BETA2*B + BETA3*C

    When TAU2 is NaN the BETA3*C term is set to 0 (Nelson-Siegel 3-param fallback).
    Output units match BETA0 (percent).
    """
    x1  = tau / t1
    A   = np.where(x1 == 0.0, 1.0, (1.0 - np.exp(-x1)) / x1)
    B   = A - np.exp(-x1)
    x2  = np.where(np.isnan(t2), 0.0, tau / t2)
    A2  = np.where(x2 == 0.0, 1.0, (1.0 - np.exp(-x2)) / x2)
    C   = A2 - np.exp(-x2)
    B3C = np.where(np.isnan(t2), 0.0, b3 * C)
    return b0 + b1 * A + b2 * B + B3C


# ── Restrict to date window and drop rows missing required parameters ─────────
df = gsw_raw.loc[GSW_START:GSW_END,
                 ['BETA0', 'BETA1', 'BETA2', 'BETA3', 'TAU1', 'TAU2']].dropna(
                 subset=['BETA0', 'BETA1', 'BETA2', 'BETA3', 'TAU1'])

b0, b1, b2, b3 = df['BETA0'].values, df['BETA1'].values, df['BETA2'].values, df['BETA3'].values
t1, t2         = df['TAU1'].values, df['TAU2'].values

# ── Compute daily GSW yields at all 11 maturities ────────────────────────────
print("Computing Svensson zero-coupon yields at target maturities...")
yields_daily = pd.DataFrame(index=df.index, columns=GSW_LABELS, dtype=float)
for tau, col in zip(GSW_MATURITIES, GSW_LABELS):
    yields_daily[col] = svensson_yield(tau, b0, b1, b2, b3, t1, t2)

print(f"Daily GSW yields shape : {yields_daily.shape}")
print(f"Date range             : {yields_daily.index.min().date()} – {yields_daily.index.max().date()}")
print(f"Missing values / col   : {yields_daily.isna().sum().to_dict()}")

# ── Sanity check 1: raw parameters → computed yields (manual verify) ─────────
print("\n── Sanity check 1: raw params → computed yields ──────────────────────────")
sample_idx = [0, len(df) // 2, len(df) - 1]
for i in sample_idx:
    d   = df.index[i]
    row = df.iloc[i]
    tau2_str = f"{row.TAU2:.4f}" if pd.notna(row.TAU2) else "NaN (Nelson-Siegel)"
    print(f"\n  {d.date()}: BETA0={row.BETA0:.4f}  BETA1={row.BETA1:.4f}  "
          f"BETA2={row.BETA2:.4f}  BETA3={row.BETA3:.4f}  "
          f"TAU1={row.TAU1:.4f}  TAU2={tau2_str}")
    for tau_chk, col_chk in zip([1, 5, 10], ['1Y', '5Y', '10Y']):
        print(f"    y({col_chk:>3s}) = {yields_daily.loc[d, col_chk]:.4f}%")

# ── Sanity check 2: computed yields vs SVENY* pre-computed columns ────────────
print("\n── Sanity check 2: computed vs SVENY* reference yields ───────────────────")
sveny_map  = {'SVENY01': '1Y', 'SVENY02': '2Y', 'SVENY03': '3Y',
              'SVENY05': '5Y', 'SVENY07': '7Y', 'SVENY10': '10Y'}
ref_cols   = list(sveny_map.keys())
our_cols   = list(sveny_map.values())

ref_daily  = gsw_raw.loc[GSW_START:GSW_END, ref_cols].dropna()
our_check  = yields_daily.loc[ref_daily.index, our_cols]
ref_check  = ref_daily.copy()
ref_check.columns = our_cols

diffs = (our_check - ref_check).abs()
valid = diffs.values[~np.isnan(diffs.values)]
print(f"  Max  |computed - SVENY*| : {valid.max():.6f} pp")
print(f"  Mean |computed - SVENY*| : {valid.mean():.6f} pp")
print("  (< 0.001 confirms Svensson formula is correctly implemented)")

# ── Resample to monthly (month-end mean, forward-fill residual gaps) ──────────
yields_monthly = yields_daily.resample('ME').mean().ffill().bfill()
yields_monthly = yields_monthly.dropna(how='all')
print(f"\nMonthly GSW yields shape : {yields_monthly.shape}")

# ── Save ──────────────────────────────────────────────────────────────────────
yields_daily.to_csv(GSW_DAILY_OUT)
yields_monthly.to_csv(GSW_MONTHLY_OUT)
print(f"Saved : {GSW_DAILY_OUT}")
print(f"Saved : {GSW_MONTHLY_OUT}")
print(f"Columns: {list(yields_daily.columns)}")

Computing Svensson zero-coupon yields at target maturities...
Daily GSW yields shape : (11481, 11)
Date range             : 1980-01-02 – 2025-12-31
Missing values / col   : {'1M': 0, '3M': 0, '6M': 0, '1Y': 0, '2Y': 0, '3Y': 0, '5Y': 0, '7Y': 0, '10Y': 0, '20Y': 0, '30Y': 0}

── Sanity check 1: raw params → computed yields ──────────────────────────

  1980-01-02: BETA0=10.3001  BETA1=2.7114  BETA2=-621.4268  BETA3=617.0601  TAU1=2.0704  TAU2=2.0725
    y( 1Y) = 11.6037%
    y( 5Y) = 10.1118%
    y(10Y) = 10.0960%

  2003-01-22: BETA0=0.7940  BETA1=0.7060  BETA2=-1.5574  BETA3=15.5312  TAU1=0.6490  TAU2=13.1825
    y( 1Y) = 1.2535%
    y( 5Y) = 2.9809%
    y(10Y) = 4.3504%

  2025-12-31: BETA0=2.3287  BETA1=1.3732  BETA2=-0.0001  BETA3=9.1299  TAU1=1.2166  TAU2=18.0866
    y( 1Y) = 3.5082%
    y( 5Y) = 3.7092%
    y(10Y) = 4.2568%

── Sanity check 2: computed vs SVENY* reference yields ───────────────────
  Max  |computed - SVENY*| : 0.000335 pp
  Mean |computed - SVENY*| : 0.000018 pp